# Energy model review and experiment

The candidate improved development scores, but did not beat the current v3 architecture on reserved confirmation rows. Keep the current competition best as the default. The experiment is useful for testing how missing inputs and distribution shift affect predictions.

See `model_review.md` for the code review. The hidden test labels are unavailable; the 3.20 leaderboard target has not been verified.


In [ ]:
import json
import pandas as pd
from energy_model import ROOT, EnergyModel, write_submission

train = pd.read_csv(ROOT / 'train.csv')
test = pd.read_csv(ROOT / 'test.csv')
print(f'{len(train):,} labelled rows; {len(test):,} test rows')


## Read the independent confirmation results

Settings were selected using 4-fold validation on 6,400 development rows. The 1,600 confirmation rows were reserved before selection. Test masks were applied to complete confirmation rows, across ten trials. The weighted check approximates the test building/event mix; it is not an official test score.


In [ ]:
results = json.loads((ROOT / 'validation_results.json').read_text())
scores = pd.DataFrame({
    'Current v3 architecture': {
        'Ordinary confirmation': results['baseline']['ordinary_validation']['rmse'],
        'Test missing patterns': results['baseline']['test_masks_rmse'],
        'Building/event weighted check': results['baseline']['building_event_weighted_rmse'],
    },
    'Experimental ensemble': {
        'Ordinary confirmation': results['candidate']['ordinary_validation']['rmse'],
        'Test missing patterns': results['candidate']['test_masks_rmse'],
        'Building/event weighted check': results['candidate']['building_event_weighted_rmse'],
    },
})
scores.round(4)


## Reproduce the confirmation experiment

This uses the fixed settings in `model_config.json` and the same reserved split. It fits only on the 6,400 development rows. Unlabelled test features estimate training weights; confirmation targets are used only for scoring. Running the next cell recomputes `validation_results.json` and may take a minute.


In [ ]:
from validate_energy_model import validate
reproduced = validate()


## Generate the experimental submission

The next cell fits on all 8,000 labelled rows and writes `candidate_v4.csv`. It is a candidate to investigate, not a demonstrated improvement over the existing competition best. It uses the existing environment and submission format.


In [ ]:
submission = write_submission(output_path=ROOT / 'candidate_v4.csv')
assert len(submission) == len(test)
assert submission['id'].equals(test['id'])
assert submission['prediction'].notna().all()
submission.head()


## Why the candidate was tried

It combines the current v3 architecture with models trained directly for each pattern of available numeric inputs. Building effects, richer daily cycles, categorical residual trees, and modest distribution weighting were tested. Development results are in `experiments/`; independent confirmation did not establish a gain.
